# Model-based RL with a neural model: MPC on Pendulum

`dyna_q.ipynb` kept the model in a table: one entry per `(s, a)`. Pendulum's state is continuous (`[cos θ, sin θ, θ̇]`), so no two states are ever exactly the same, and a table can't hold it. The model becomes a **neural network** `f(s, a) → s'`, fit on transitions the same way the critic was fit on targets.

**How to use the model** is the other change. Dyna used it to make extra training data for Q. Here there is **no policy and no value function at all**, only planning:

1. At every step, imagine many candidate futures with the model: "if I apply these 30 torques, where does the pendulum go, and how much reward do I collect?"
2. Take the **first** action of the best candidate, then throw the rest of the plan away.
3. Next step, plan again from the real new state.

This is **model predictive control (MPC)**. Re-planning every step means the model's mistakes never get far: each plan only has to be right for one action before reality corrects it.

**Reward:** the reward function `-(θ² + 0.1·θ̇² + 0.001·u²)` is given to the planner rather than learned. Many model-based papers assume the same, and learning it would be a second small regression. The **dynamics** are what gets learned.

For comparison, from `ddpg_td3_sac.ipynb`: random play scores about -1200, and DDPG/TD3/SAC reach about -165 after 20k steps.


In [1]:
import gymnasium as gym
import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn

device = "cuda" if torch.cuda.is_available() else "cpu"
torch.manual_seed(0)
np.random.seed(0)
env = gym.make("Pendulum-v1")
env.action_space.seed(0)

0

## Step 1: learn the dynamics from random play

Collect 50 episodes (10k steps) of random torques and fit `f(s, a) → s'` by plain regression. The model has two standard details:

- **Predict the change, not the next state:** `s' = s + net(s, a)`. In 0.05 s the pendulum barely moves, so `s'` is almost `s`. Predicting the small difference is much easier than re-deriving the whole state.
- **Normalize the change:** each output is scaled by the typical size of that state's change (`scale`), so the small `cos`/`sin` moves and the large `θ̇` moves count equally in the loss.


In [2]:
def collect(policy, n_episodes, seed=0):
    """Runs policy (state -> action) for n_episodes.
    Returns (S, A, S2, returns): stacked transitions as float32 arrays, and each episode's return."""
    S, A, S2, returns = [], [], [], []
    for i in range(n_episodes):
        s, _ = env.reset(seed=seed + i)
        done, total = False, 0.0
        while not done:
            a = policy(s)
            s2, r, term, trunc, _ = env.step(a)
            S.append(s)
            A.append(a)
            S2.append(s2)
            s, total, done = s2, total + r, term or trunc
        returns.append(total)
    f = lambda x: np.array(x, np.float32)
    return f(S), f(A), f(S2), returns


S, A, S2, rand_returns = collect(lambda s: env.action_space.sample(), 50)
print(f"{len(S)} transitions, random return {np.mean(rand_returns):.0f}")
print(f"near upright (cos θ > 0.9): {(S[:, 0] > 0.9).mean():.1%} of states")
print(f"|θ̇| up to {np.abs(S[:, 2]).max():.1f} (the env's limit is 8)")

10000 transitions, random return -1228
near upright (cos θ > 0.9): 4.1% of states
|θ̇| up to 8.0 (the env's limit is 8)


Random torques can't swing the pendulum up on purpose. But each episode **starts** at a random angle, so the data still covers fast spins and a few near-upright states. The model gets to see most of the state space.


In [3]:
class Dynamics(nn.Module):
    """f(s, a) -> s', as s + scale * net(s, a)."""

    def __init__(self, scale):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(4, 256), nn.ReLU(),
            nn.Linear(256, 256),
            nn.ReLU(), nn.Linear(256, 3),
        )
        self.register_buffer("scale", torch.as_tensor(scale, dtype=torch.float32))

    def forward(self, s, a):
        return s + self.scale * self.net(
            torch.cat([s, a / 2.0], -1)
        )  # torque in [-2, 2] -> [-1, 1]


def fit(S, A, S2, epochs=60, batch_size=256, lr=1e-3):
    """Returns (model, losses): the trained Dynamics and the loss after each epoch."""
    model = Dynamics(scale=(S2 - S).std(0) + 1e-6).to(device)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    X, U, Y = (torch.as_tensor(v, device=device) for v in (S, A, S2))
    losses = []
    for _ in range(epochs):
        perm = torch.randperm(len(X), device=device)
        for i in range(0, len(X), batch_size):
            idx = perm[i : i + batch_size]
            loss = (((model(X[idx], U[idx]) - Y[idx]) / model.scale) ** 2).mean()
            opt.zero_grad()
            loss.backward()
            opt.step()
        losses.append(loss.item())
    return model, losses


model, losses = fit(S, A, S2)
print(
    f"final loss {losses[-1]:.4f} (in units of the typical change: 1.0 would be guessing 'no change')"
)

final loss 0.0006 (in units of the typical change: 1.0 would be guessing 'no change')


**One step ahead vs many.** A low training loss only says the model is good for **one** step. The planner will chain 30 of them, feeding each prediction back in as the next input, so small errors can compound.

Test both on 10 fresh random episodes the model never saw. From a real state, apply the actions that were really taken, using only the model, and compare its angle with the real one after k steps.


In [4]:
def angle(s):
    return np.arctan2(s[..., 1], s[..., 0])


tS, tA, tS2, _ = collect(lambda s: env.action_space.sample(), 10, seed=1000)
tS, tA = tS.reshape(10, 200, 3), tA.reshape(10, 200, 1)  # (episode, step, dim)

ks = [1, 5, 10, 20, 30, 50]
err = {k: [] for k in ks}
with torch.no_grad():
    for t0 in range(0, 150, 10):  # several start points per episode
        x = torch.as_tensor(tS[:, t0], device=device)
        for k in range(1, max(ks) + 1):
            x = model(
                x, torch.as_tensor(tA[:, t0 + k - 1], device=device)
            )  # model's own output fed back in
            if k in ks:
                diff = angle(x.cpu().numpy()) - angle(tS[:, t0 + k])
                err[k] += np.abs(np.angle(np.exp(1j * diff))).tolist()  # wrap to [-pi, pi]

for k in ks:
    print(f"{k:2d} steps ahead: mean angle error {np.degrees(np.mean(err[k])):5.1f}°")

 1 steps ahead: mean angle error   0.1°
 5 steps ahead: mean angle error   0.6°
10 steps ahead: mean angle error   1.9°
20 steps ahead: mean angle error   7.7°
30 steps ahead: mean angle error  13.9°
50 steps ahead: mean angle error  23.1°


**Error compounds with the horizon.** One step ahead the model is nearly exact. Over tens of steps the small errors pile up, because each prediction is built on the previous one. That's why MPC re-plans every step: only the first action of each plan is ever used, and it's chosen from the part of the plan where the model is most accurate.


## Step 2: random-shooting MPC

The simplest planner there is:

1. Sample `N` random torque sequences, each `H` steps long (uniform in `[-2, 2]`).
2. Roll all of them through the model **at once**, as a batch on the GPU, adding up the reward along the way.
3. Take the first torque of the sequence with the highest total.

No gradients and no training: the "policy" is a search, redone at every step.


In [5]:
def reward_fn(s, a):
    """Pendulum's reward, from the observation [cos θ, sin θ, θ̇] and torque, batched."""
    th = torch.atan2(s[..., 1], s[..., 0])
    return -(th**2 + 0.1 * s[..., 2] ** 2 + 0.001 * a[..., 0] ** 2)


@torch.no_grad()
def plan_returns(dynamics, s, actions):
    """Total imagined reward of each candidate. actions: (N, H, 1). Returns (N,)."""
    x = s.repeat(len(actions), 1)
    total = torch.zeros(len(actions), device=device)
    for h in range(actions.shape[1]):
        total += reward_fn(x, actions[:, h])
        x = dynamics(x, actions[:, h])
    return total


def random_shooting(dynamics, N=1000, H=30):
    """An MPC policy: state -> first torque of the best of N random H-step sequences."""

    def policy(s):
        actions = torch.rand(N, H, 1, device=device) * 4 - 2
        best = plan_returns(dynamics, torch.as_tensor(s, device=device), actions).argmax()
        return actions[best, 0].cpu().numpy()

    return policy


def evaluate(policy, n_episodes=5, seed=100):
    """Returns the episode returns of policy on n_episodes fresh starts."""
    return collect(policy, n_episodes, seed=seed)[3]


r = evaluate(random_shooting(model))
print(f"random shooting, learned model: {np.mean(r):5.0f}   episodes: {np.round(r).astype(int)}")

random shooting, learned model:  -341   episodes: [-351 -400 -356 -120 -477]


**Much better than random play (about -1200), but well short of SAC's -165.** Some episodes come in at about -120 (a clean swing-up and balance). Others are around -350.

Is the **model** to blame, or the **planner**? Test the planner on its own by giving it the **true** physics instead of the learned model. Pendulum's equations are short:

    θ̇' = clip(θ̇ + (3g/2 · sin θ + 3u) · dt, -8, 8),    θ' = θ + θ̇' · dt,    g = 10, dt = 0.05

With a perfect model, any shortfall is the planner's fault.


In [6]:
def true_dynamics(s, a):
    """Pendulum-v1's exact physics, batched, in the same [cos θ, sin θ, θ̇] form as the model."""
    th, thd = torch.atan2(s[..., 1], s[..., 0]), s[..., 2]
    u = a[..., 0].clamp(-2, 2)
    thd = (thd + (3 * 10 / 2 * torch.sin(th) + 3.0 * u) * 0.05).clamp(-8, 8)
    th = th + thd * 0.05
    return torch.stack([torch.cos(th), torch.sin(th), thd], -1)


for N, H in [(1000, 30), (4000, 30), (1000, 60)]:
    for name, dyn in [("learned", model), ("TRUE", true_dynamics)]:
        r = evaluate(random_shooting(dyn, N, H))
        print(
            f"N={N:4d} H={H}  {name:7s} model: {np.mean(r):5.0f}   episodes: {np.round(r).astype(int)}"
        )

N=1000 H=30  learned model:  -245   episodes: [-241 -391 -241 -120 -234]


N=1000 H=30  TRUE    model:  -314   episodes: [-361 -496 -242 -120 -352]


N=4000 H=30  learned model:  -267   episodes: [-245 -498 -235 -120 -237]


N=4000 H=30  TRUE    model:  -245   episodes: [-240 -387 -243 -119 -236]


N=1000 H=60  learned model:  -728   episodes: [-582 -759 -989 -613 -697]


N=1000 H=60  TRUE    model:  -601   episodes: [-475 -853 -486 -362 -829]


**The planner is the bottleneck, not the model.**

- **Learned ≈ true.** Even with the exact physics, random shooting lands at about -250 to -310. The model's error doesn't explain the gap to -165. (The two "learned, N=1000, H=30" runs above gave -341 and -245 on the same 5 starts. Random shooting draws fresh random sequences every step, so it's noisy enough that 5 episodes can't separate learned from true.)
- **More samples help only a little** (N=4000: about -250).
- **A longer horizon makes it much worse, even with perfect physics** (H=60: -600 to -730). That's surprising if you think the problem is not seeing far enough ahead. The real problem is the search: a sequence of 60 independent random torques almost never does anything coherent, like a sustained push in one direction. Over 60 steps all 1000 candidates are bad, and the best of them is mostly luck.

So the model is good enough already. What's missing is a smarter way to **search** over action sequences, one that improves its candidates instead of drawing them blindly.


## Step 3: CEM, a smarter search

Random shooting guesses blindly. **CEM** (cross-entropy method) guesses, then learns from its own best guesses:

1. Sample 500 plans from a Gaussian (start: mean 0, std 1 per step).
2. Score them in the model and keep the best 50.
3. Refit the Gaussian to those 50, then sample again from it.
4. Repeat 5 times, then take the first action of the final mean.

**Warm start:** the next step begins from this step's plan, shifted by one, instead of from scratch.


In [7]:
def cem(dynamics, H=30, N=500, n_elite=50, iters=5):
    """An MPC policy: state -> first torque of the plan CEM converges to.
    Make a fresh one per episode: its plan carries over from step to step."""
    plan = torch.zeros(H, 1, device=device)  # mean of the Gaussian over plans, kept between steps

    def policy(s):
        nonlocal plan
        s = torch.as_tensor(s, device=device)
        mean, std = plan, torch.ones_like(plan)
        for _ in range(iters):
            actions = (mean + std * torch.randn(N, H, 1, device=device)).clamp(-2, 2)
            elite = actions[plan_returns(dynamics, s, actions).topk(n_elite).indices]
            mean, std = elite.mean(0), elite.std(0) + 1e-3  # refit to the best 50
        plan = torch.cat([mean[1:], torch.zeros(1, 1, device=device)])  # warm start: shift by one
        return mean[0].cpu().numpy()

    return policy


for name, dyn in [("learned", model), ("TRUE", true_dynamics)]:
    # a fresh planner per episode, so no plan leaks from one episode into the next
    r = [evaluate(cem(dyn), n_episodes=1, seed=100 + i)[0] for i in range(10)]
    print(
        f"CEM, {name:7s} model: {np.mean(r):5.0f} ± {np.std(r):3.0f}   episodes: {np.round(r).astype(int)}"
    )

CEM, learned model:  -195 ±  75   episodes: [-122 -280 -251 -120 -246 -123 -317 -123 -240 -128]


CEM, TRUE    model:  -189 ±  87   episodes: [-121 -338 -256 -119 -120 -121 -326 -122 -238 -127]


**CEM: about -195, vs random shooting's -250 to -340.** The learned model does as well as the true physics (-189), so the model is still not the limit.

**Episodes split in two:** about -120 (clean swing-up) or -240 to -330 (hard starts). The true physics has the same split, so those hard starts are on the planner.

**Same 10 starts as `ddpg_td3_sac.ipynb`, where SAC scored about -165.** CEM gets within about 30 of it, with no policy network and only 10k random steps of data.